# ShellForge selected rank-16 inference
Run in your own free Colab GPU session. This is an experimental session, not always-on hosting. No paid resources are requested. The CPU smoke in the repository was verified; this notebook must be run to establish GPU evidence. No shell command generated by the model is executed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/ShellForge')
if not repo.exists():
    subprocess.run(['git','clone','--branch','feat/shellforge-product-v2','https://github.com/rupeshhh007/ShellForge.git',str(repo)],check=True)
subprocess.run(['python','-m','pip','install','-r',str(repo/'requirements-model.txt'),'-r',str(repo/'services/api/requirements.txt')],check=True)

In [ ]:
import torch, shutil, sys, tempfile
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > free GPU; do not purchase compute.'
source=Path('/content/drive/MyDrive/ShellForge/full-data-v1/trial-2')
working=Path(tempfile.mkdtemp(prefix='shellforge-adapter-'))
for name in ['adapter_config.json','adapter_model.safetensors','tokenizer_config.json']:
    shutil.copy2(source/'adapter'/name,working/name)
shutil.copy2(source/'training_manifest.json',working/'training_manifest.json')
sys.path.insert(0,str(repo))
from services.inference.runtime import validate_adapter
validate_adapter(working)

In [ ]:
import os
os.chdir(repo)
output=Path('/content/drive/MyDrive/ShellForge/full-data-v1/selected-gpu-smoke.json')
subprocess.run(['python','-m','scripts.smoke_selected_model','--adapter',str(working),'--output',str(output)],check=True)
print(output.read_text())

Optional serving: follow docs/product/INFERENCE.md with a private token and an authenticated HTTPS origin. Colab can disconnect and is not a production SLA. Never put credentials in the notebook, outputs, repository or browser bundle. Do not claim a live website connection from this saved smoke alone.